In [2]:
import pandas as pd
from tweetple import TweetPle
import sys
sys.path.insert(0, '../../src/utils')
from funcs import *
from tqdm import tqdm
sys.path.insert(0, '../../src/utils')
from general import *

In [3]:
base = f'../../data/03-experiment/SA/baseline/'
path_tw = base + '00-raw/influencers/tweets/'
path_likes = f'{base}00-raw/influencers/likes/'

In [3]:
tweets = pd.read_parquet('../../data/03-experiment/KE/baseline/01-preprocess/influencers/tweets_batch2.parquet')

In [4]:
followers_sa = pd.read_parquet('../../data/03-experiment/SA/baseline/01-preprocess/influencers/followers_ties_batch2.parquet')

In [56]:
followers_grouped_n_sa = followers_sa[['influencer_id', 'follower_id']].groupby('influencer_id').count()
followers_grouped_n_sa.rename({'follower_id': 'followers'}, axis=1, inplace=True)

followers_grouped_sa = followers_sa.groupby('influencer_id').sum()
followers_grouped_sa = followers_grouped_sa.merge(followers_grouped_n_sa, on='influencer_id', how='left')
followers_grouped_sa = followers_grouped_sa[['followers','strong', 'weak']]
followers_grouped_sa['absent'] = followers_grouped_sa['followers'] - followers_grouped_sa['strong'] - followers_grouped_sa['weak']

<ipython-input-56-9ee54b530135>:4: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  followers_grouped_sa = followers_sa.groupby('influencer_id').sum()


In [5]:
followers_ke = pd.read_parquet('../../data/03-experiment/KE/baseline/01-preprocess/influencers/followers_ties_batch2.parquet')

In [58]:
followers_grouped_n = followers_ke[['influencer_id', 'follower_id']].groupby('influencer_id').count()
followers_grouped_n.rename({'follower_id': 'followers'}, axis=1, inplace=True)

followers_grouped_ke = followers_ke.groupby('influencer_id').sum()
followers_grouped_ke = followers_grouped_ke.merge(followers_grouped_n, on='influencer_id', how='left')

followers_grouped_ke = followers_grouped_ke[['followers','strong', 'weak']]
followers_grouped_ke['absent'] = followers_grouped_ke['followers'] - followers_grouped_ke['strong'] - followers_grouped_ke['weak']

<ipython-input-58-925037d57192>:4: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  followers_grouped_ke = followers_ke.groupby('influencer_id').sum()


In [59]:
rand_vars_ke = pd.read_parquet('../../data/randomization/KE/02-variables/variables_batch2.parquet')
rand_vars_sa = pd.read_parquet('../../data/randomization/SA/02-variables/variables_batch2.parquet')

In [60]:
followers_grouped_ke.rename(columns = {'followers':'n_followers',
                            'strong':'n_strong',
                            'weak':'n_weak',
                            'absent':'n_absent'}, inplace = True)
followers_grouped_sa.rename(columns = {'followers':'n_followers',
                            'strong':'n_strong',
                            'weak':'n_weak',
                            'absent':'n_absent'}, inplace = True)

In [61]:
followers_grouped_sa.reset_index(inplace=True)
followers_grouped_ke.reset_index(inplace=True)

In [62]:
followers_grouped_ke.rename(columns = {'influencer_id':'author_id'}, inplace = True)
followers_grouped_sa.rename(columns = {'influencer_id':'author_id'}, inplace = True)

In [63]:
followers_grouped_ke['author_id'] = followers_grouped_ke['author_id'].astype(str)
followers_grouped_sa['author_id'] = followers_grouped_sa['author_id'].astype(str)
rand_vars_ke['author_id'] = rand_vars_ke['author_id'].astype(str)
rand_vars_sa['author_id'] = rand_vars_sa['author_id'].astype(str)

In [64]:
rand_vars_ke = pd.merge(rand_vars_ke,followers_grouped_ke,on='author_id',how='left')
rand_vars_sa = pd.merge(rand_vars_sa,followers_grouped_sa,on='author_id',how='left')

In [65]:
rand_vars_sa.columns

Index(['Unnamed: 0', 'author_id', 'n_tweets.fc', 'n_tweets.na', 'n_tweets',
       'handle', 'impression_count_min', 'impression_count_mean',
       'impression_count_max', 'impression_count_sum', 'impression_count_std',
       'like_count_min', 'like_count_mean', 'like_count_max', 'like_count_sum',
       'like_count_std', 'quote_count_min', 'quote_count_mean',
       'quote_count_max', 'quote_count_sum', 'quote_count_std',
       'reply_count_min', 'reply_count_mean', 'reply_count_max',
       'reply_count_sum', 'reply_count_std', 'retweet_count_min',
       'retweet_count_mean', 'retweet_count_max', 'retweet_count_sum',
       'retweet_count_std', 'interactions_count_sum', 'location', 'created_at',
       'description', 'name', 'url', 'username', 'verified', 'followers_count',
       'following_count', 'listed_count', 'tweet_count', 'days_old_account',
       'n_followers', 'n_strong', 'n_weak', 'n_absent'],
      dtype='object')

In [66]:
# Remove timezone from columns if needed
rand_vars_sa['created_at'] = rand_vars_sa['created_at'].dt.tz_localize(None)
rand_vars_ke['created_at'] = rand_vars_ke['created_at'].dt.tz_localize(None)

In [67]:
rand_vars_sa = rand_vars_sa.drop('Unnamed: 0', axis=1)

In [68]:
rand_vars_ke.to_excel('../../data/randomization/KE/02-variables/variables_batch2.xlsx')
rand_vars_sa.to_excel('../../data/randomization/SA/02-variables/variables_batch2.xlsx')

##### In case of days_old needed

In [19]:
from datetime import date
today = date.today()
rand_vars_ke['date_today'] = today.strftime("%Y-%m-%d")
rand_vars_sa['date_today'] = today.strftime("%Y-%m-%d")

In [77]:
rand_vars_ke['created_at'] = rand_vars_ke['created_at'].apply(pd.to_datetime) #if conversion required
rand_vars_ke['date_today'] = rand_vars_ke['date_today'].apply(pd.to_datetime) #if conversion required
rand_vars_ke['days_old_account'] = (rand_vars_ke['date_today'] - rand_vars_ke['created_at']).dt.days

rand_vars_sa['created_at'] = rand_vars_sa['created_at'].apply(pd.to_datetime) #if conversion required
rand_vars_sa['date_today'] = rand_vars_sa['date_today'].apply(pd.to_datetime) #if conversion required
rand_vars_sa['days_old_account'] = (rand_vars_sa['date_today'] - rand_vars_sa['created_at']).dt.days

In [79]:
rand_vars_ke.to_excel('../../data/randomization/KE/02-variables/variables.xlsx')
rand_vars_sa.to_excel('../../data/randomization/SA/02-variables/variables.xlsx')

### Followers Randomization

In [7]:
followers_sa = pd.read_parquet('../../data/03-experiment/SA/baseline/01-preprocess/influencers/followers_ties_batch2.parquet')

In [8]:
# Fixing ties: if strong then weak = 0
followers_sa['weak'] = np.where(followers_sa['strong']== 1, 0, followers_sa['weak'])

In [71]:
followers_grouped_n_sa = followers_sa[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_n_sa.rename({'influencer_id': 'n_following'}, axis=1, inplace=True)

followers_grouped_sa = followers_sa.groupby('follower_id').sum()
followers_grouped_sa = followers_grouped_sa.merge(followers_grouped_n_sa, on='follower_id', how='left')
followers_grouped_sa = followers_grouped_sa[['n_following','strong', 'weak']]
followers_grouped_sa['absent'] = followers_grouped_sa['n_following'] - followers_grouped_sa['strong'] - followers_grouped_sa['weak']

<ipython-input-71-e4df46c9e32c>:4: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  followers_grouped_sa = followers_sa.groupby('follower_id').sum()


In [72]:
followers_ke = pd.read_parquet('../../data/03-experiment/KE/baseline/01-preprocess/influencers/followers_ties_batch2.parquet')

In [73]:
followers_ke['weak'] = np.where(followers_ke['strong']== 1, 0, followers_ke['weak'])

In [74]:
followers_grouped_n_ke = followers_ke[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_n_ke.rename({'influencer_id': 'n_following'}, axis=1, inplace=True)

followers_grouped_ke = followers_ke.groupby('follower_id').sum()
followers_grouped_ke = followers_grouped_ke.merge(followers_grouped_n_ke, on='follower_id', how='left')
followers_grouped_ke = followers_grouped_ke[['n_following','strong', 'weak']]
followers_grouped_ke['absent'] = followers_grouped_ke['n_following'] - followers_grouped_ke['strong'] - followers_grouped_ke['weak']

<ipython-input-74-afb371ce1d9e>:4: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  followers_grouped_ke = followers_ke.groupby('follower_id').sum()


In [75]:
followers_grouped_sa.reset_index(inplace=True)
followers_grouped_ke.reset_index(inplace=True)
followers_grouped_ke.rename(columns = {'followers':'n_followers',
                            'strong':'n_strong',
                            'weak':'n_weak',
                            'absent':'n_absent'}, inplace = True)
followers_grouped_sa.rename(columns = {'followers':'n_followers',
                            'strong':'n_strong',
                            'weak':'n_weak',
                            'absent':'n_absent'}, inplace = True)

followers_grouped_ke['follower_id'] = followers_grouped_ke['follower_id'].astype(str)
followers_grouped_sa['follower_id'] = followers_grouped_sa['follower_id'].astype(str)

In [8]:
b_path = f'../../data/01-characterize/followers/KE/00-raw/'
file = 'integrate/followers_batch2.parquet.gzip'
info_followers_ke = pd.read_parquet(f'{b_path}{file}')

b_path = f'../../data/01-characterize/followers/SA/00-raw/'
file = 'integrate/followers_batch2.parquet.gzip'
info_followers_sa = pd.read_parquet(f'{b_path}{file}')

In [78]:
from ast import literal_eval


info_followers_sa = info_followers_sa.join(
      pd.json_normalize(info_followers_sa['public_metrics'])
 ).drop('public_metrics',axis=1)

info_followers_ke = info_followers_ke.join(
      pd.json_normalize(info_followers_ke['public_metrics'])
 ).drop('public_metrics',axis=1)

In [79]:
info_followers_sa.drop_duplicates('id', keep = 'last', inplace=True)
info_followers_ke.drop_duplicates('id', keep = 'last', inplace=True)

In [80]:
info_followers_ke.reset_index(drop=True, inplace=True)
info_followers_sa.reset_index(drop=True, inplace=True)

In [81]:
ids_ke = list(info_followers_ke['id'].astype(str))
ids_sa = list(info_followers_sa['id'].astype(str))
bearer_token = "AAAAAAAAAAAAAAAAAAAAAFpgZAEAAAAAbJS59UWzipi32ixd7LHtXov9olo%3D7gxD8Afshgj4munMXHLU08jzRdTpsAh4RZqq7VBofq1wAvkx1T"

In [82]:
info_followers_ke.rename(columns={'id':'follower_id',
                                  'public_metrics.followers_count':'followers_count',
                                  'public_metrics.following_count': 'following_count',
                                  'public_metrics.tweet_count':'tweet_count',
                                  'public_metrics.listed_count':'listed_count',}, inplace=True)
info_followers_sa.rename(columns={'id':'follower_id',
                                  'public_metrics.followers_count':'followers_count',
                                  'public_metrics.following_count': 'following_count',
                                  'public_metrics.tweet_count':'tweet_count',
                                  'public_metrics.listed_count':'listed_count',}, inplace=True)

In [83]:
info_followers_ke = info_followers_ke.merge(followers_grouped_ke,on='follower_id',how='left')
info_followers_sa = info_followers_sa.merge(followers_grouped_sa,on='follower_id',how='left')

In [84]:
info_followers_ke.to_parquet('../../data/randomization/KE/02-variables/variables_followers_batch2.parquet')
info_followers_sa.to_parquet('../../data/randomization/SA/02-variables/variables_followers_batch2.parquet')

In [ ]:
#### ???????

In [85]:
treatment_ke = pd.read_excel('../../data/randomization/KE/03-assignment/output/RandomizedTwitterSampleKE_batch2.xlsx')
treatment_sa = pd.read_excel('../../data/randomization/SA/03-assignment/output/RandomizedTwitterSampleSA_batch2.xlsx')

treatment_ke['author_id'] = treatment_ke['author_id'].astype(str)
treatment_sa['author_id'] = treatment_sa['author_id'].astype(str)

In [86]:
control_ke = list(treatment_ke[treatment_ke['treatment']==0].author_id)
control_sa = list(treatment_sa[treatment_sa['treatment']==0].author_id)

treat_ke = list(treatment_ke[treatment_ke['treatment']==1].author_id)
treat_sa = list(treatment_sa[treatment_sa['treatment']==1].author_id)

In [87]:
info_followers_ke = pd.read_parquet('../../data/randomization/KE/02-variables/variables_followers_batch2.parquet')
info_followers_sa = pd.read_parquet('../../data/randomization/SA/02-variables/variables_followers_batch2.parquet')



In [88]:
followers_sa = pd.read_parquet('../../data/03-experiment/SA/baseline/01-preprocess/influencers/followers_ties_batch2.parquet')
followers_ke =  pd.read_parquet('../../data/03-experiment/KE/baseline/01-preprocess/influencers/followers_ties_batch2.parquet')

followers_ke['weak'] = np.where(followers_ke['strong']== 1, 0, followers_ke['weak'])
followers_sa['weak'] = np.where(followers_sa['strong']== 1, 0, followers_sa['weak'])

followers_sa = followers_sa[followers_sa['influencer_id'].isin(treat_sa)]
followers_ke = followers_ke[followers_ke['influencer_id'].isin(treat_ke)]

In [89]:
followers_grouped_t_sa = followers_sa[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_t_sa.rename({'influencer_id': 'n_following_treated'}, axis=1, inplace=True)


followers_grouped_sa_t = followers_sa.groupby('follower_id').sum()
followers_grouped_sa_t = followers_grouped_sa_t.merge(followers_grouped_t_sa, on='follower_id', how='left')
followers_grouped_sa_t = followers_grouped_sa_t[['n_following_treated','strong', 'weak']]
followers_grouped_sa_t['absent'] = followers_grouped_sa_t['n_following_treated'] - followers_grouped_sa_t['strong'] - followers_grouped_sa_t['weak']



<ipython-input-89-9f32412d0fb9>:5: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  followers_grouped_sa_t = followers_sa.groupby('follower_id').sum()


In [90]:
followers_grouped_t_ke = followers_ke[followers_ke['influencer_id'].isin(treat_ke)]
followers_grouped_t_ke = followers_grouped_t_ke[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_t_ke.rename({'influencer_id': 'n_following_treated'}, axis=1, inplace=True)


followers_grouped_ke_t = followers_ke[followers_ke['influencer_id'].isin(treat_ke)]
followers_grouped_ke_t = followers_grouped_ke_t.groupby('follower_id').sum()
followers_grouped_ke_t = followers_grouped_ke_t.merge(followers_grouped_t_ke, on='follower_id', how='left')
followers_grouped_ke_t = followers_grouped_ke_t[['n_following_treated','strong', 'weak']]
followers_grouped_ke_t['absent'] = followers_grouped_ke_t['n_following_treated'] - followers_grouped_ke_t['strong'] - followers_grouped_ke_t['weak']




<ipython-input-90-ec48111dec97>:7: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  followers_grouped_ke_t = followers_grouped_ke_t.groupby('follower_id').sum()


In [91]:
followers_grouped_sa_t.reset_index(inplace=True)
followers_grouped_ke_t.reset_index(inplace=True)
followers_grouped_ke_t.rename(columns = {'followers':'n_followers_treated',
                            'strong':'n_strong_treated',
                            'weak':'n_weak_treated',
                            'absent':'n_absent_treated'}, inplace = True)
followers_grouped_sa_t.rename(columns = {'followers':'n_followers_treated',
                            'strong':'n_strong_treated',
                            'weak':'n_weak_treated',
                            'absent':'n_absent_treated'}, inplace = True)

followers_grouped_ke_t['follower_id'] = followers_grouped_ke_t['follower_id'].astype(str)
followers_grouped_sa_t['follower_id'] = followers_grouped_sa_t['follower_id'].astype(str)

In [92]:
info_followers_ke = info_followers_ke.merge(followers_grouped_ke_t,on='follower_id',how='left')
info_followers_sa = info_followers_sa.merge(followers_grouped_sa_t,on='follower_id',how='left')

In [92]:
## control

In [63]:
#info_followers_ke = pd.read_parquet('../../data/randomization/KE/02-variables/variables_followers.parquet')
#info_followers_sa = pd.read_parquet('../../data/randomization/SA/02-variables/variables_followers.parquet')

In [94]:
followers_sa = pd.read_parquet('../../data/03-experiment/SA/baseline/01-preprocess/influencers/followers_ties_batch2.parquet')
followers_ke =  pd.read_parquet('../../data/03-experiment/KE/baseline/01-preprocess/influencers/followers_ties_batch2.parquet')

followers_ke['weak'] = np.where(followers_ke['strong']== 1, 0, followers_ke['weak'])
followers_sa['weak'] = np.where(followers_sa['strong']== 1, 0, followers_sa['weak'])

followers_sa = followers_sa[followers_sa['influencer_id'].isin(control_sa)]
followers_ke = followers_ke[followers_ke['influencer_id'].isin(control_ke)]

followers_grouped_c_sa = followers_sa[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_c_sa.rename({'influencer_id': 'n_following_control'}, axis=1, inplace=True)


followers_grouped_sa_c = followers_sa.groupby('follower_id').sum()
followers_grouped_sa_c = followers_grouped_sa_c.merge(followers_grouped_c_sa, on='follower_id', how='left')
followers_grouped_sa_c = followers_grouped_sa_c[['n_following_control','strong', 'weak']]
followers_grouped_sa_c['absent'] = followers_grouped_sa_c['n_following_control'] - followers_grouped_sa_c['strong'] - followers_grouped_sa_c['weak']

followers_grouped_c_ke = followers_ke[followers_ke['influencer_id'].isin(control_ke)]
followers_grouped_c_ke = followers_grouped_c_ke[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_c_ke.rename({'influencer_id': 'n_following_control'}, axis=1, inplace=True)


followers_grouped_ke_c = followers_ke[followers_ke['influencer_id'].isin(control_ke)]
followers_grouped_ke_c = followers_grouped_ke_c.groupby('follower_id').sum()
followers_grouped_ke_c = followers_grouped_ke_c.merge(followers_grouped_c_ke, on='follower_id', how='left')
followers_grouped_ke_c = followers_grouped_ke_c[['n_following_control','strong', 'weak']]
followers_grouped_ke_c['absent'] = followers_grouped_ke_c['n_following_control'] - followers_grouped_ke_c['strong'] - followers_grouped_ke_c['weak']


followers_grouped_sa_c.reset_index(inplace=True)
followers_grouped_ke_c.reset_index(inplace=True)
followers_grouped_ke_c.rename(columns = {'followers':'n_followers_control',
                            'strong':'n_strong_control',
                            'weak':'n_weak_control',
                            'absent':'n_absent_control'}, inplace = True)
followers_grouped_sa_c.rename(columns = {'followers':'n_followers_control',
                            'strong':'n_strong_control',
                            'weak':'n_weak_control',
                            'absent':'n_absent_control'}, inplace = True)

followers_grouped_ke_c['follower_id'] = followers_grouped_ke_c['follower_id'].astype(str)
followers_grouped_sa_c['follower_id'] = followers_grouped_sa_c['follower_id'].astype(str)

<ipython-input-94-781b4acc1e0c>:14: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  followers_grouped_sa_c = followers_sa.groupby('follower_id').sum()
<ipython-input-94-781b4acc1e0c>:25: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  followers_grouped_ke_c = followers_grouped_ke_c.groupby('follower_id').sum()


In [95]:
info_followers_ke = info_followers_ke.merge(followers_grouped_ke_c,on='follower_id',how='left')
info_followers_sa = info_followers_sa.merge(followers_grouped_sa_c,on='follower_id',how='left')

In [96]:
info_followers_ke.drop('author_id_following', axis=1, inplace=True)
info_followers_sa.drop('author_id_following', axis=1, inplace=True)

In [98]:
info_followers_ke["n_following_treated"] = info_followers_ke["n_following_treated"].fillna(0)
info_followers_ke["n_strong_treated"] = info_followers_ke["n_strong_treated"].fillna(0)
info_followers_ke["n_weak_treated"] = info_followers_ke["n_weak_treated"].fillna(0)
info_followers_ke["n_absent_treated"] = info_followers_ke["n_absent_treated"].fillna(0)
info_followers_ke["n_following_control"] = info_followers_ke["n_following_control"].fillna(0)
info_followers_ke["n_strong_control"] = info_followers_ke["n_strong_control"].fillna(0)
info_followers_ke["n_weak_control"] = info_followers_ke["n_weak_control"].fillna(0)
info_followers_ke["n_absent_control"] = info_followers_ke["n_absent_control"].fillna(0)


In [99]:
info_followers_sa["n_following_treated"] = info_followers_sa["n_following_treated"].fillna(0)
info_followers_sa["n_strong_treated"] = info_followers_sa["n_strong_treated"].fillna(0)
info_followers_sa["n_weak_treated"] = info_followers_sa["n_weak_treated"].fillna(0)
info_followers_sa["n_absent_treated"] = info_followers_sa["n_absent_treated"].fillna(0)
info_followers_sa["n_following_control"] = info_followers_sa["n_following_control"].fillna(0)
info_followers_sa["n_strong_control"] = info_followers_sa["n_strong_control"].fillna(0)
info_followers_sa["n_weak_control"] = info_followers_sa["n_weak_control"].fillna(0)
info_followers_sa["n_absent_control"] = info_followers_sa["n_absent_control"].fillna(0)

In [102]:
info_followers_ke.to_parquet('../../data/randomization/KE/02-variables/variables_followers_batch2.parquet')
info_followers_sa.to_parquet('../../data/randomization/SA/02-variables/variables_followers_batch2.parquet')

In [104]:
import pytz
from datetime import datetime, timedelta
import pandas as pd
from datetime import date
today = datetime.now(tz=pytz.UTC)
info_followers_ke['date_today'] = today
info_followers_sa['date_today'] = today

In [105]:
info_followers_ke['created_at'] = info_followers_ke['created_at'].apply(pd.to_datetime) #if conversion required
info_followers_ke['date_today'] = info_followers_ke['date_today'].apply(pd.to_datetime) #if conversion required
info_followers_ke['days_old_account'] = (info_followers_ke['date_today'] - info_followers_ke['created_at']).dt.days

info_followers_sa['created_at'] = info_followers_sa['created_at'].apply(pd.to_datetime) #if conversion required
info_followers_sa['date_today'] = info_followers_sa['date_today'].apply(pd.to_datetime) #if conversion required
info_followers_sa['days_old_account'] = (info_followers_sa['date_today'] - info_followers_sa['created_at']).dt.days

In [106]:
ids = list(info_followers_ke[info_followers_ke['followers_count']<0]['follower_id'].astype(str))

In [139]:
pd.set_option('display.max_columns', None)
info_followers_ke[info_followers_ke['followers_count']<0]

,created_at,description,entities,follower_id,location,name,pinned_tweet_id,profile_image_url,protected,url,username,verified,date_consulted,response,withheld,followers_count,following_count,listed_count,tweet_count,n_following,n_strong,n_weak,n_absent,n_following_treated,n_strong_treated,n_weak_treated,n_absent_treated,n_following_control,n_strong_control,n_weak_control,n_absent_control,date_today,days_old_account


In [113]:
import tweetple

from tweetple import TweetPle

# Bearer token accesible via Twitter Developer Academic Research Track
bearer_token='AAAAAAAAAAAAAAAAAAAAAFpgZAEAAAAAbJS59UWzipi32ixd7LHtXov9olo%3D7gxD8Afshgj4munMXHLU08jzRdTpsAh4RZqq7VBofq1wAvkx1T'

# List of handle ids
ids = list(info_followers_ke[info_followers_ke['followers_count']<0].follower_id)


missing_info = TweetPle.TweepleStreamer(ids, bearer_token).user_lookup()

0it [00:00, ?it/s]/Users/joaquinbarrutia/miniforge3/envs/env_tf/lib/python3.9/site-packages/tweetple/TweetPle.py:61: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_stats = df_stats.append(
1it [00:01,  1.30s/it]


In [138]:
#corregir wrong entries

info_followers_ke.loc[174632,'followers_count'] = 0
info_followers_ke.loc[174632,'following_count'] = 61
info_followers_ke.loc[174632,'listed_count'] = 0
info_followers_ke.loc[174632,'tweet_count'] = 0

In [140]:
info_followers_ke.to_parquet('../../data/randomization/KE/02-variables/variables_followers_batch2.parquet')
info_followers_sa.to_parquet('../../data/randomization/SA/02-variables/variables_followers_batch2.parquet')

In [63]:
followers_sa = pd.read_parquet('../../data/03-experiment/KE/baseline/01-preprocess/influencers/followers_ties.parquet')

In [72]:
ids = pd.read_parquet(f'../../data/randomization/KE/04-stratification/integrate/followers_randomized_batch2.parquet')

In [73]:
info_followers_sa= pd.read_parquet('../../data/randomization/KE/02-variables/variables_followers_batch2.parquet')

In [74]:
ids = ids.merge(info_followers_sa, on='follower_id', how='left')

In [75]:
r = ids[['follower_id','username_x', 'username_y', 'n_strong', 'n_weak', 'n_absent']]

In [76]:
len(r[((r['n_weak']==0) & (r['n_strong']==0))])

4709

In [80]:
r[((r['n_weak']==0) & (r['n_strong']==0))]

,follower_id,username_x,username_y,n_strong,n_weak,n_absent
22780,1251551208508133377,jmmyben254,jmmyben254,0,0,1
22782,1252695460965474313,AwuorMauleen,AwuorMauleen,0,0,1
22783,1252717673680637953,MrTittoh,MrTittoh,0,0,1
22785,1253051671694934022,tityb3,tityb3,0,0,1
22786,1253783890248032266,OchokoNick,OchokoNick,0,0,2
...,...,...,...,...,...,...
29321,998243290058248193,ontha_mmm,ontha_mmm,0,0,1
29322,99912456,kelvin_tn07,kelvin_tn07,0,0,1
29323,999301444422569984,TestBot115,TestBot115,0,0,2
29324,999315917350821888,KhanyaeD,KhanyaeD,0,0,2


In [78]:
len(r[~((r['n_weak']==0) & (r['n_strong']==0))])

35485

In [71]:
5754/(22887+5754)

0.20090080653608464